# 04 — Finetune DENSE base (n50k) on two-hop composition  ← RUN THIS PAIR FIRST
n50k is the cleaner test: dense **memorized** the facts (~63% recall) and split externalized them, so both arms have fact-access — any composition gap is about the substrate.

Runtime → **GPU (A100)**, Run all. **Start this FIRST**; once Cell 3 builds the corpus, start `05_split_n50k`. Reuses the same `compose_v1` corpus as the n800k pair (same seed 1234).
Local-disk output; eval results copy to Drive; re-Run-all to restart if dropped.


In [ ]:
# Cell 1 — mount Drive, clone branch, drop in finetune.py + configs
from google.colab import drive; drive.mount('/content/drive')
%cd /content
!rm -rf Memory-Split && git clone -q --branch feat/ood-2-hop https://github.com/syz2026/Memory-Split.git
%cd /content/Memory-Split
FT = '/content/drive/MyDrive/ms/compose-finetune'
!cp {FT}/finetune.py .
!mkdir -p configs && cp {FT}/configs/compose_dense_ft.yaml {FT}/configs/compose_split_ft.yaml configs/
!pip -q install tiktoken pyyaml numpy 2>/dev/null
import torch; print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only!')

In [ ]:
# Cell 2 — parameters (already set for this notebook; nothing to edit)
ARM   = 'dense'
BASE  = 'n50k'
ALIGNED_SEED = 1234        # confirmed from n800k data build (report.json)
SMOKE = False

DRIVE = '/content/drive/MyDrive/ms'
import os, glob
cands = sorted(glob.glob(f'{DRIVE}/snapshots/{ARM}_{BASE}_step*.pt'))
assert cands, f'no base snapshot at {DRIVE}/snapshots/{ARM}_{BASE}_step*.pt  (upload it to Drive)'
INIT_FROM = cands[-1]
TRAINER   = 'v2' if ARM == 'split' else 'v1'
tag  = 'smoke' if SMOKE else 'v1'
ALN  = f'{DRIVE}/data/compose_{tag}'          # aligned corpus on Drive (shared by both notebooks)
NOV  = f'{DRIVE}/data/compose_novel_{tag}'    # novel-entity OOD (different seed)
# Heavy run output (20 snapshots x ~619MB + ckpt) goes to Colab LOCAL disk (~80GB free),
# NOT Drive (15GB free tier) -> avoids filling Drive / crashing mid-run.
OUT_DIR = f'/content/runs/{ARM}_{BASE}_ft' + ('_smoke' if SMOKE else '')
# Only the small eval results (summary/figures/json) are copied back to Drive:
RESULTS_DRIVE = f'{DRIVE}/results/{ARM}_{BASE}_ft' + ('_smoke' if SMOKE else '')
print('arm', ARM, '| trainer', TRAINER, '| smoke', SMOKE)
print('init from', INIT_FROM)
print('out (local)', OUT_DIR, '| results ->', RESULTS_DRIVE)

In [ ]:
# Cell 3 — build compose ONCE (aligned, full) + a novel-entity OOD set (eval-only, tiny). Shared via Drive; skipped if present.
NENT, TOK, NEV = (500, 3_000_000, 100) if SMOKE else (10000, 600_000_000, 1000)
NOV_TOK = TOK if SMOKE else 20_000_000   # novel set is eval-only -> tiny train.bin (saves Drive space)
if not os.path.exists(f'{ALN}/report.json'):
    !python scripts/build_compose.py --out {ALN} --n-entities {NENT} --held-frac 0.2 --total-tokens {TOK} --n-eval {NEV} --seed {ALIGNED_SEED}
else:
    print('aligned corpus already built:', ALN)
if not os.path.exists(f'{NOV}/report.json'):
    !python scripts/build_compose.py --out {NOV} --n-entities {NENT} --held-frac 0.2 --total-tokens {NOV_TOK} --n-eval {NEV} --seed {ALIGNED_SEED + 777}
else:
    print('novel corpus already built:', NOV)

In [ ]:
# Cell 4 — finetune (writes ckpts/snapshots to Drive; re-run this cell to RESUME if the session drops)
!rm -rf data && mkdir -p data && ln -sf {ALN} data/compose_v1   # config's relative path -> Drive corpus
CFG = f'configs/compose_{ARM}_ft.yaml'
extra = '--max-steps 40' if SMOKE else ''
!python finetune.py --config {CFG} --trainer {TRAINER} --init-from {INIT_FROM} --out-dir {OUT_DIR} --run-id {ARM}_{BASE}_ft {extra}

In [ ]:
# Cell 5 — evaluate: aligned OOD (+ OOD-vs-step curve = keep-best) and novel-entity OOD
import shutil, os, json
ARMFLAG = '--arm split' if ARM == 'split' else ''
ALN_OUT = f'{OUT_DIR}/eval_aligned'    # separate dirs so novel doesn't overwrite aligned
NOV_OUT = f'{OUT_DIR}/eval_novel'
print('=== aligned OOD (P_held) + OOD-vs-step curve ===')
!python scripts/run_compose_eval.py --run {OUT_DIR} --data {ALN} --out {ALN_OUT} {ARMFLAG}
print('=== novel-entity OOD (unseen people) ===')
!python scripts/run_compose_eval.py --run {OUT_DIR} --data {NOV} --out {NOV_OUT} {ARMFLAG}
print('--- aligned summary (the primary result) ---')
!cat {ALN_OUT}/summary.md 2>/dev/null | head -60

# Persist the SMALL eval outputs (summary/figures/json) to Drive so they survive the session.
os.makedirs(RESULTS_DRIVE, exist_ok=True)
for sub in ['eval_aligned', 'eval_novel']:
    s = f'{OUT_DIR}/{sub}'
    if os.path.exists(s):
        shutil.copytree(s, f'{RESULTS_DRIVE}/{sub}', dirs_exist_ok=True)
print('saved eval results to', RESULTS_DRIVE)

# Keep the BEST-by-OOD checkpoint (read from the aligned curve) on Drive (~619MB).
res = json.load(open(f'{ALN_OUT}/results.json'))
curve = res.get('curve', [])
if curve:
    best = max(curve, key=lambda c: c['ood_acc'])   # peak OOD accuracy over training
    snap = f'{OUT_DIR}/snapshots/step{best["step"]:07d}.pt'
    if os.path.exists(snap):
        dst = f'{RESULTS_DRIVE}/best_snapshot_step{best["step"]:07d}_ood{best["ood_acc"]:.3f}.pt'
        shutil.copy(snap, dst)
        print(f'kept best-by-OOD snapshot: step {best["step"]}  OOD {best["ood_acc"]:.1%}  -> {dst}')
    else:
        print('best snapshot file missing:', snap)